# 02 — Detection Rule Performance Analysis

Analyses precision, FP burden, and Pareto distribution of detection rules.

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import matplotlib.pyplot as plt
import numpy as np
from src.config import load_config
from src.statistics import compute_rule_precision_ci, compute_pareto

cfg = load_config()
db_path = cfg['paths']['database']

In [ ]:
# Rule performance scorecard
con = duckdb.connect(db_path, read_only=True)
con.execute("""
    SELECT detection_rule, alert_volume, tp_count, fp_count,
           precision, fp_rate, volume_share, fp_volume_share, tp_volume_share
    FROM dim_rule_performance
    ORDER BY fp_count DESC
""").fetchdf()

In [ ]:
# Precision with Wilson confidence intervals
ci_data = compute_rule_precision_ci(db_path)
import pandas as pd
pd.DataFrame(ci_data)

In [ ]:
# Three ranking perspectives
print('=== Ranking A: Raw FP Count ===')
display(con.execute("""
    SELECT detection_rule, fp_count, alert_volume, precision
    FROM dim_rule_performance ORDER BY fp_count DESC LIMIT 10
""").fetchdf())

print('\n=== Ranking B: FP Rate (min 30 alerts) ===')
display(con.execute("""
    SELECT detection_rule, fp_rate, fp_count, alert_volume
    FROM dim_rule_performance WHERE alert_volume >= 30 ORDER BY fp_rate DESC LIMIT 10
""").fetchdf())

print('\n=== Ranking C: Volume-Weighted FP Burden ===')
display(con.execute("""
    SELECT detection_rule, fp_count, fp_rate,
           ROUND(fp_count * fp_rate, 2) AS burden_score, alert_volume
    FROM dim_rule_performance ORDER BY burden_score DESC LIMIT 10
""").fetchdf())

In [ ]:
# Pareto analysis
pareto = compute_pareto(db_path)
for threshold in [50, 80, 90]:
    rules_at = [p for p in pareto if p['cum_fp_pct'] <= threshold]
    count = len(rules_at) + 1 if rules_at else 1
    actual_pct = pareto[min(count-1, len(pareto)-1)]['cum_fp_pct']
    print(f'{count} rule(s) account for ~{actual_pct}% of FP volume')

In [ ]:
con.close()